# Feature basis — 20 features over 50 choice sets

**Goal.** Build a single shared, interpretable feature basis (20 features) that applies to *every* retained choice set, then score 50 sampled sets on it. This is the basis we'll later score over all retained sets and feed into a linear preference model.

**Two stages:**
1. **Discover** the 20-feature basis from a small diverse subsample (12 sets shown to Claude with full prompts, responses, and annotator feedback). Output: 20 generic, scoreable features with rubrics. Saved to `feature_basis.json`.
2. **Score** all 50 study sets on the discovered basis. Output: a long-form dataframe with one row per `(choice_set, response_position, feature)`. Saved to `feature_scores_long.parquet`.

**Cost discipline.** The scoring system prompt (= the 20 feature definitions + rubrics) is `cache_control`-marked so per-set scoring calls read it from cache at ~10% input price. For an eventual scale-up to all retained sets, swap the sequential loop for the Batches API.

**Inputs.** `empirics_communityalignment/choice_sets_ge10.parquet` (produced by `secondpass_CAanalysis.ipynb`).

**Out of scope here.** Feature canonicalization across multiple basis-discovery runs, fitting per-annotator weights, anything model-side.

## 1. Setup

In [8]:
from __future__ import annotations

import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import anthropic
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "empirics_communityalignment"
INPUT_PARQUET = DATA_DIR / "choice_sets_ge10.parquet"
OUT_BASIS_JSON = DATA_DIR / "feature_basis.json"
OUT_SCORES_PARQUET = DATA_DIR / "feature_scores_long.parquet"

# Sampling / basis size
SEED = 20260426
N_STUDY_SETS = 50
N_DISCOVERY_SETS = 12
N_FEATURES = 20
MODEL_ID = "claude-opus-4-7"

if not INPUT_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing {INPUT_PARQUET}. Run secondpass_CAanalysis.ipynb first."
    )

annotations_df = pd.read_parquet(INPUT_PARQUET)
n_sets_total = annotations_df["choice_set_id"].nunique()
print(
    f"Loaded {len(annotations_df):,} retained annotations "
    f"across {n_sets_total:,} choice sets."
)
if n_sets_total < N_STUDY_SETS:
    raise RuntimeError(
        f"Only {n_sets_total} retained sets available; lower N_STUDY_SETS or rerun secondpass with a smaller threshold."
    )

client = anthropic.Anthropic()

Loaded 14,231 retained annotations across 845 choice sets.


## 2. Sample 50 study sets and build per-set context

Random sample of 50 retained choice sets (seeded). For each sampled set we extract the prompt, the four responses, the choice distribution, and a sample of annotator feedback. This `build_set_context` helper feeds both stage 1 (basis discovery) and stage 2 (per-set scoring).

In [9]:
rng = np.random.default_rng(SEED)
all_set_ids = annotations_df["choice_set_id"].drop_duplicates().tolist()
study_set_ids = list(rng.choice(all_set_ids, size=N_STUDY_SETS, replace=False))
discovery_set_ids = list(rng.choice(study_set_ids, size=N_DISCOVERY_SETS, replace=False))

print(f"Sampled {len(study_set_ids)} study sets (of {len(all_set_ids):,} retained).")
print(f"Of those, {len(discovery_set_ids)} go to stage-1 discovery.")

Sampled 50 study sets (of 845 retained).
Of those, 12 go to stage-1 discovery.


In [10]:
def build_set_context(
    annotations_df: pd.DataFrame,
    choice_set_id: str,
    n_feedback_samples: int = 6,
) -> dict:
    """Pull prompt, 4 responses, choice distribution, and feedback sample for one set."""
    sub = annotations_df[annotations_df["choice_set_id"] == choice_set_id]
    if len(sub) == 0:
        raise KeyError(f"choice_set_id not found: {choice_set_id}")
    prompt_text = sub["prompt_text"].iloc[0]
    responses = {f"response_{i}": sub[f"response_{i}"].iloc[0] for i in range(1, 5)}
    counts = sub["chosen_response"].value_counts(dropna=False).to_dict()
    fb_pool = sub[sub["first_turn_feedback"].notna()][
        ["chosen_response", "first_turn_feedback"]
    ]
    n_take = min(n_feedback_samples, len(fb_pool))
    fb_sample = (
        fb_pool.sample(n=n_take, random_state=SEED).reset_index(drop=True)
        if n_take > 0
        else fb_pool.reset_index(drop=True)
    )
    return {
        "choice_set_id": choice_set_id,
        "prompt_text": prompt_text,
        "responses": responses,
        "choice_counts": counts,
        "feedback_sample": fb_sample,
    }


study_contexts: dict[str, dict] = {
    cs_id: build_set_context(annotations_df, cs_id) for cs_id in study_set_ids
}
print(f"Built context for {len(study_contexts)} sets.")

Built context for 50 sets.


## 3. Stage 1 — Discover the 20-feature basis

Show Claude all 12 discovery sets at once and ask for 20 features that:
- generalize across topics (the basis will be applied to many sets we *haven't* shown);
- vary across responses (a feature where all 4 score the same on most sets is useless);
- capture preference-relevant dimensions, including ones annotators are likely to *disagree* on weighting;
- are linearizable: scoreable on a single 1-5 Likert per response;
- cover diverse aspects (substance, style, structure, stance, register, etc.) so the basis isn't redundant.

Output is structured (Pydantic), with name + definition + scoring rubric (1/3/5 anchors) + rationale + aspect for each feature.

In [11]:
class FeatureDefinition(BaseModel):
    name: str = Field(..., description="Short snake_case identifier (e.g. 'specificity', 'tone_warmth').")
    definition: str = Field(..., description="One-sentence operational definition that a non-expert annotator could apply.")
    scoring_rubric: str = Field(
        ...,
        description=(
            "Concrete anchors for what 1, 3, and 5 look like (e.g. '1 = no specifics, only abstract claims; "
            "3 = some examples but partial; 5 = densely specific with named entities, numbers, steps')."
        ),
    )
    rationale: str = Field(..., description="Why annotators are likely to care about this AND disagree on its weight.")
    aspect: str = Field(
        ...,
        description=(
            "High-level category, e.g. 'substance', 'style', 'structure', 'stance', 'register', 'reader_orientation'."
        ),
    )


class FeatureBasis(BaseModel):
    features: list[FeatureDefinition] = Field(
        ...,
        description=(
            f"Exactly {N_FEATURES} features. They must NOT be near-duplicates and must cover diverse aspects."
        ),
    )

In [12]:
def format_set_for_discovery(
    ctx: dict,
    max_response_chars: int = 700,
    max_feedback_chars: int = 250,
    n_feedback: int = 4,
) -> str:
    """Compact rendering for stage-1 input — many sets fit in a single message."""
    def trunc(text: str, n: int) -> str:
        text = str(text)
        return text if len(text) <= n else text[:n] + " ..."

    parts = [
        f"=== CHOICE SET {ctx['choice_set_id'][:10]} ===",
        f"PROMPT: {ctx['prompt_text']}",
        f"CHOICE COUNTS: {ctx['choice_counts']}",
        "RESPONSES:",
    ]
    for k, v in ctx["responses"].items():
        parts.append(f"  [{k}] {trunc(v, max_response_chars)}")
    fb = ctx["feedback_sample"].head(n_feedback)
    if len(fb):
        parts.append("FEEDBACK:")
        for _, row in fb.iterrows():
            parts.append(
                f"  [chose {row['chosen_response']}] {trunc(row['first_turn_feedback'], max_feedback_chars)}"
            )
    return "\n".join(parts)


discovery_block = "\n\n".join(
    format_set_for_discovery(study_contexts[cs_id]) for cs_id in discovery_set_ids
)
print(f"Discovery user-message block: {len(discovery_block):,} chars")

Discovery user-message block: 43,531 chars


In [13]:
DISCOVERY_SYSTEM_PROMPT = f"""\
You are designing an INTERPRETABLE FEATURE BASIS for a linear preference model over 4-way response choices.

The basis you produce will be applied to hundreds of choice sets on many different prompts. So features must:
1. GENERALIZE — be scoreable on responses to ANY prompt, not just the example sets shown.
2. VARY — produce score variance across the 4 responses for a wide range of sets.
3. DRIVE PREFERENCE — capture dimensions humans actually weigh when picking, including ones reasonable annotators might WEIGHT DIFFERENTLY (this is what we want to recover).
4. BE LINEARIZABLE — scoreable on a single 1-5 Likert per response.
5. COVER DIVERSE ASPECTS — span categories such as substantive content, style/register, structure, evaluative stance, hedging/certainty, breadth/scope, reader-orientation, tone, etc. The {N_FEATURES} features should NOT be near-duplicates of each other.

For each feature provide:
- name (snake_case)
- definition (one operational sentence)
- scoring_rubric (concrete anchors at 1, 3, and 5 — what does each level look like)
- rationale (why annotators care AND why they might disagree on weighting)
- aspect (high-level category)

The annotator FEEDBACK samples in the user message are the strongest signal for what humans actually noticed and articulated as their reason for choosing — use them to ground the basis in real preference signal, not just what an LLM might find theoretically interesting.

Produce exactly {N_FEATURES} features.
"""

DISCOVERY_USER_MESSAGE = f"""\
Below are {N_DISCOVERY_SETS} example choice sets sampled from a community alignment dataset. Each shows the prompt, four candidate responses, the choice distribution among annotators, and a sample of rationales annotators wrote when they made their choice.

Use these to inform your feature basis, but DO NOT overfit to them — the basis will be applied to many more choice sets covering many more topics.

{discovery_block}

Now propose {N_FEATURES} features per the system instructions."""

print(f"System prompt: {len(DISCOVERY_SYSTEM_PROMPT):,} chars")
print(f"User message:  {len(DISCOVERY_USER_MESSAGE):,} chars")

System prompt: 1,456 chars
User message:  43,975 chars


In [14]:
discovery_response = client.messages.parse(
    model=MODEL_ID,
    max_tokens=16000,
    thinking={"type": "adaptive"},
    system=DISCOVERY_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": DISCOVERY_USER_MESSAGE}],
    output_format=FeatureBasis,
)
u = discovery_response.usage
print(
    f"Discovery usage: input={u.input_tokens:,} output={u.output_tokens:,} "
    f"cache_read={u.cache_read_input_tokens:,} cache_create={u.cache_creation_input_tokens:,}"
)
basis: FeatureBasis = discovery_response.parsed_output
print(f"Got {len(basis.features)} features.")
if len(basis.features) != N_FEATURES:
    print(f"WARNING: expected {N_FEATURES}, got {len(basis.features)}. Continuing with what we have.")

# Persist the basis so re-runs (and downstream notebooks) don't need a fresh discovery call.
OUT_BASIS_JSON.write_text(basis.model_dump_json(indent=2))
print(f"Saved feature basis to {OUT_BASIS_JSON}")

Discovery usage: input=17,995 output=3,592 cache_read=0 cache_create=0
Got 20 features.
Saved feature basis to /Users/michellesi/Desktop/harvard/distortion/empirics_communityalignment/feature_basis.json


## 4. Display the discovered basis

In [15]:
basis_df = pd.DataFrame([f.model_dump() for f in basis.features])
display(basis_df[["name", "aspect", "definition"]])

print("\nAspect distribution:")
print(basis_df["aspect"].value_counts())

,name,aspect,definition
0,format_match_to_request,structure,"How well the response's format matches the format explicitly or implicitly requested by the prompt (e.g., monologue, note, bio, proposal, list)."
1,specificity_of_detail,substance,"Density of concrete particulars such as named entities, numbers, places, steps, and examples versus abstract generalities."
2,conciseness,style,Whether the response delivers its content efficiently without unnecessary padding or verbosity.
3,comprehensiveness,substance,"Breadth of relevant subtopics, options, or considerations covered in the response."
4,structural_organization,structure,"Use of clear visual structure such as headings, numbered lists, bullet points, or paragraph breaks that aid scanning."
5,emotional_warmth,tone,"Degree of warmth, affection, or emotional resonance in tone, especially for personal/expressive prompts."
6,reader_orientation,reader_orientation,"How directly the response addresses and engages the reader/user, rather than being abstract or impersonal."
7,asks_clarifying_questions,reader_orientation,Whether the response asks for missing information needed to give a properly tailored answer.
8,actionability,substance,"Whether the response gives the user something they can immediately use or act on (template, ranked list, concrete steps)."
9,vivid_imagery,style,"Use of sensory, visual, or evocative language that paints a picture for the reader."



Aspect distribution:
substance             6
style                 5
reader_orientation    3
structure             2
stance                2
tone                  1
register              1
Name: aspect, dtype: int64


In [16]:
for f in basis.features:
    print(f"## {f.name}  [{f.aspect}]")
    print(f"   definition: {f.definition}")
    print(f"   rubric    : {f.scoring_rubric}")
    print(f"   rationale : {f.rationale}")
    print()

## format_match_to_request  [structure]
   definition: How well the response's format matches the format explicitly or implicitly requested by the prompt (e.g., monologue, note, bio, proposal, list).
   rubric    : 1 = wrong format entirely (e.g., narration when monologue requested); 3 = partially matches the requested form but with notable deviations; 5 = exactly the requested form, including conventions (salutation for a note, first-person present for a monologue, etc.).
   rationale : Annotators frequently call out format adherence as decisive (e.g., 'it's actually a monologue', 'written as a note'). Some weight strict format-fit highly, others tolerate substitutions.

## specificity_of_detail  [substance]
   definition: Density of concrete particulars such as named entities, numbers, places, steps, and examples versus abstract generalities.
   rubric    : 1 = only abstract claims, no named entities or figures; 3 = a few specifics mixed with generic statements; 5 = densely specific 

## 5. Stage 2 — Score all 50 study sets on the basis

Per-set scoring call structure:
- **System prompt** = the 20 feature definitions and rubrics (stable, `cache_control`-marked → ~10% input cost on subsequent calls).
- **User message** = one choice set's prompt + 4 responses + choice distribution + feedback samples.
- **Output** = a list of 20 `FeatureScore` entries, each with `feature_name` and a `ResponseScores` (1-5 Likert per response).

Sequential loop with a progress bar. Failures are caught per-set so one bad call doesn't kill the run.

In [17]:
class ResponseScores(BaseModel):
    response_1: int = Field(..., ge=1, le=5)
    response_2: int = Field(..., ge=1, le=5)
    response_3: int = Field(..., ge=1, le=5)
    response_4: int = Field(..., ge=1, le=5)


class FeatureScore(BaseModel):
    feature_name: str = Field(..., description="Must exactly match one of the basis feature names.")
    scores: ResponseScores


class ChoiceSetScores(BaseModel):
    feature_scores: list[FeatureScore] = Field(
        ...,
        description=(
            "One entry per feature in the basis, IN THE ORDER LISTED in the system prompt. "
            "Use the full 1-5 range when responses meaningfully differ; do not bunch values."
        ),
    )

In [18]:
def build_scoring_system_prompt(basis: FeatureBasis) -> str:
    parts = [
        "You score four candidate responses against a fixed feature basis. For each feature, score every "
        "one of the four responses on a 1-5 Likert scale, following the feature's rubric. Use the FULL 1-5 "
        "range when responses meaningfully span the dimension; do not bunch all four on the same value.\n\n",
        "Return one entry per feature, IN THE ORDER LISTED BELOW. Each entry's `feature_name` must exactly "
        "match the name shown.\n\n",
        "FEATURE BASIS:\n",
    ]
    for i, f in enumerate(basis.features, 1):
        parts.append(f"\n{i}. {f.name}  [{f.aspect}]\n")
        parts.append(f"   Definition: {f.definition}\n")
        parts.append(f"   Rubric    : {f.scoring_rubric}\n")
    return "".join(parts)


SCORING_SYSTEM = build_scoring_system_prompt(basis)
SCORING_SYSTEM_BLOCKS = [
    {"type": "text", "text": SCORING_SYSTEM, "cache_control": {"type": "ephemeral"}}
]
print(f"Scoring system prompt: {len(SCORING_SYSTEM):,} chars (cacheable)")

Scoring system prompt: 7,165 chars (cacheable)


In [19]:
def format_feedback_for_scoring(fb_df: pd.DataFrame, max_chars: int = 400) -> str:
    if len(fb_df) == 0:
        return "  (no annotator feedback available)"
    lines = []
    for i, row in fb_df.iterrows():
        text = str(row["first_turn_feedback"])
        if len(text) > max_chars:
            text = text[:max_chars] + " ..."
        lines.append(f"  [{i+1}] (chose {row['chosen_response']}) {text}")
    return "\n".join(lines)


def build_scoring_user_message(ctx: dict) -> str:
    counts_block = "\n".join(
        f"  {code}: {cnt}"
        for code, cnt in sorted(ctx["choice_counts"].items(), key=lambda kv: str(kv[0]))
    )
    resp_block = "\n\n".join(f"[{k}]\n{v}" for k, v in ctx["responses"].items())
    fb_block = format_feedback_for_scoring(ctx["feedback_sample"])
    return f"""\
PROMPT:
{ctx['prompt_text']}

ANNOTATOR CHOICE DISTRIBUTION:
{counts_block}

CANDIDATE RESPONSES:

{resp_block}

ANNOTATOR FEEDBACK:
{fb_block}

Score every feature in the basis on every response."""


def score_one_set(ctx: dict) -> tuple[ChoiceSetScores, anthropic.types.Usage]:
    response = client.messages.parse(
        model=MODEL_ID,
        max_tokens=8000,
        thinking={"type": "adaptive"},
        system=SCORING_SYSTEM_BLOCKS,
        messages=[{"role": "user", "content": build_scoring_user_message(ctx)}],
        output_format=ChoiceSetScores,
    )
    return response.parsed_output, response.usage

In [20]:
results: dict[str, ChoiceSetScores] = {}
errors: dict[str, str] = {}
totals = {"input": 0, "output": 0, "cache_read": 0, "cache_create": 0}

for cs_id in tqdm(study_set_ids, desc="scoring sets"):
    try:
        scored, usage = score_one_set(study_contexts[cs_id])
        results[cs_id] = scored
        totals["input"] += usage.input_tokens
        totals["output"] += usage.output_tokens
        totals["cache_read"] += usage.cache_read_input_tokens
        totals["cache_create"] += usage.cache_creation_input_tokens
    except Exception as exc:  # noqa: BLE001
        errors[cs_id] = f"{type(exc).__name__}: {exc}"

print(f"\nScored: {len(results)} of {len(study_set_ids)}.")
print(f"Errors: {len(errors)}")
if errors:
    for cs_id, msg in list(errors.items())[:5]:
        print(f"  {cs_id}: {msg}")
print(
    f"Token totals: input={totals['input']:,} output={totals['output']:,} "
    f"cache_read={totals['cache_read']:,} cache_create={totals['cache_create']:,}"
)

scoring sets:   0%|          | 0/50 [00:00<?, ?it/s]


Scored: 50 of 50.
Errors: 0
Token totals: input=88,497 output=49,900 cache_read=165,914 cache_create=3,386


## 6. Build the long-form score table

One row per `(choice_set_id, feature_name, response_position)`. Validates that the names returned by Claude match the basis exactly, and that every (set × feature × response) cell is present.

In [21]:
basis_names = [f.name for f in basis.features]
basis_name_set = set(basis_names)

rows = []
name_mismatches: dict[str, set] = {}
for cs_id, scored in results.items():
    seen_names = set()
    for fs in scored.feature_scores:
        if fs.feature_name not in basis_name_set:
            name_mismatches.setdefault(cs_id, set()).add(fs.feature_name)
            continue
        seen_names.add(fs.feature_name)
        for resp_col in ("response_1", "response_2", "response_3", "response_4"):
            rows.append(
                {
                    "choice_set_id": cs_id,
                    "feature_name": fs.feature_name,
                    "response_position": resp_col,
                    "score": int(getattr(fs.scores, resp_col)),
                }
            )
    missing = basis_name_set - seen_names
    if missing:
        name_mismatches.setdefault(cs_id, set()).update(f"MISSING:{m}" for m in missing)

scores_long_df = pd.DataFrame(rows)
print(f"Long-form rows: {len(scores_long_df):,} (expected ≈ {len(results) * len(basis_names) * 4:,})")
if name_mismatches:
    print(f"\nName mismatches in {len(name_mismatches)} sets:")
    for cs_id, names in list(name_mismatches.items())[:5]:
        print(f"  {cs_id}: {sorted(names)[:5]}{'...' if len(names) > 5 else ''}")

scores_long_df.to_parquet(OUT_SCORES_PARQUET, index=False)
print(f"\nSaved {OUT_SCORES_PARQUET}")
scores_long_df.head(8)

Long-form rows: 4,000 (expected ≈ 4,000)

Saved /Users/michellesi/Desktop/harvard/distortion/empirics_communityalignment/feature_scores_long.parquet


,choice_set_id,feature_name,response_position,score
0,4a055ec45d71791d9ecf3670155acf9578d2214f,format_match_to_request,response_1,2
1,4a055ec45d71791d9ecf3670155acf9578d2214f,format_match_to_request,response_2,2
2,4a055ec45d71791d9ecf3670155acf9578d2214f,format_match_to_request,response_3,2
3,4a055ec45d71791d9ecf3670155acf9578d2214f,format_match_to_request,response_4,4
4,4a055ec45d71791d9ecf3670155acf9578d2214f,specificity_of_detail,response_1,1
5,4a055ec45d71791d9ecf3670155acf9578d2214f,specificity_of_detail,response_2,1
6,4a055ec45d71791d9ecf3670155acf9578d2214f,specificity_of_detail,response_3,1
7,4a055ec45d71791d9ecf3670155acf9578d2214f,specificity_of_detail,response_4,3


## 7. Audit — does each feature actually vary?

A feature whose 4 response scores are nearly identical across most sets is dead weight in a linear-preference model. For each feature, compute the within-set variance of scores across the 4 responses, then summarize across study sets.

In [22]:
within_set_var = (
    scores_long_df.groupby(["choice_set_id", "feature_name"])["score"]
    .var()
    .reset_index(name="within_set_variance")
)
feature_variance_summary = (
    within_set_var.groupby("feature_name")["within_set_variance"]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "mean_var", "median": "median_var", "min": "min_var", "max": "max_var"})
    .sort_values("mean_var", ascending=False)
)
print("Per-feature within-set variance summary (high mean = feature genuinely discriminates):")
display(feature_variance_summary)

Per-feature within-set variance summary (high mean = feature genuinely discriminates):


,feature_name,mean_var,median_var,min_var,max_var
18,structural_organization,2.215000,3.125000,0.0,4.000000
8,format_match_to_request,1.038333,1.000000,0.0,3.333333
0,actionability,1.030000,0.958333,0.0,4.000000
3,comprehensiveness,0.928333,0.666667,0.0,2.250000
14,prompt_constraint_fidelity,0.811667,0.666667,0.0,3.333333
17,specificity_of_detail,0.805000,0.666667,0.0,3.000000
2,balance_of_options,0.776667,0.333333,0.0,3.666667
4,conciseness,0.703333,0.666667,0.0,2.000000
15,reader_orientation,0.651667,0.333333,0.0,3.583333
6,explanation_depth,0.601667,0.333333,0.0,2.250000


In [23]:
# Spot-check one scored set — compare the per-feature winner against the empirical favorite.
spot_id = next(iter(results))
spot_scores = scores_long_df[scores_long_df["choice_set_id"] == spot_id]
spot_pivot = (
    spot_scores.pivot(index="feature_name", columns="response_position", values="score")
    [["response_1", "response_2", "response_3", "response_4"]]
)
spot_pivot["top_response"] = spot_pivot.idxmax(axis=1)
spot_pivot["variance"] = spot_pivot[["response_1", "response_2", "response_3", "response_4"]].var(axis=1)

ctx = study_contexts[spot_id]
emp_top_resp, emp_top_count = max(
    (kv for kv in ctx["choice_counts"].items() if isinstance(kv[0], str)),
    key=lambda kv: kv[1],
)
emp_total = sum(v for v in ctx["choice_counts"].values() if isinstance(v, int))
print(f"Spot-check choice_set_id: {spot_id}")
print(f"Empirical favorite among annotators: {emp_top_resp} ({emp_top_count}/{emp_total} votes)")
print("\nPrompt:", textwrap.shorten(str(ctx["prompt_text"]), width=120, placeholder=" ..."))
print("\nPer-feature scores and which response 'wins' on each feature:")
display(spot_pivot.sort_values("variance", ascending=False))

Spot-check choice_set_id: 4a055ec45d71791d9ecf3670155acf9578d2214f
Empirical favorite among annotators: response_d (9/12 votes)

Prompt: संयुक्त राज्य में स्पीकईज़ी वाला एक अनूठा रेस्टोरेंट कौन-सा है?

Per-feature scores and which response 'wins' on each feature:


response_position,response_1,response_2,response_3,response_4,top_response,variance
feature_name,,,,,,
prompt_constraint_fidelity,2,1,2,4,response_4,1.583333
actionability,1,1,1,3,response_4,1.000000
originality,1,1,1,3,response_4,1.000000
specificity_of_detail,1,1,1,3,response_4,1.000000
format_match_to_request,2,2,2,4,response_4,1.000000
comprehensiveness,2,1,2,3,response_4,0.666667
register_appropriateness,3,3,3,4,response_4,0.250000
reader_orientation,2,2,2,3,response_4,0.250000
persuasive_or_marketing_appeal,2,2,2,3,response_4,0.250000


## 8. Notes for scaling to all retained sets

- **Reuse the basis.** `feature_basis.json` is the artifact. Re-run only stage 2 against the full retained set list — no need to rediscover.
- **Switch to Batches API.** Sequential calls work for 50; for ~hundreds-to-thousands of sets, wrap the per-set call in `client.messages.batches.create(...)` for 50% pricing. Same `messages.parse` shape works inside batch requests.
- **Cache TTL.** The system prompt uses 5-minute ephemeral caching. For a multi-hour batch, switch to `"ttl": "1h"` (cache write is 2× input price for 1h, but break-even is ~3 reads).
- **Drop low-variance features.** The summary in §7 flags features whose within-set variance is consistently near zero. Drop them or merge with related features and rerun, since they can't help discriminate choices in a linear model.
- **Multi-pass denoising (optional).** A single Likert from one LLM call is noisy. For the production basis, score each set with multiple seeds/orderings and reconcile (mean / median / mode), or score with a separate verifier model.
- **Basis stability across discovery runs.** A shared basis derived from one 12-set discovery sample isn't necessarily robust. To check: run discovery 2-3 more times with different sub-samples, embed the resulting feature definitions, and inspect overlap. Stable themes survive; idiosyncratic ones don't.